# 07b — Clustering en espacio de embeddings (validación semántica)

**Objetivo:** correr un clustering independiente sobre el **espacio de embeddings** (`embeddings_personas.csv`, embedding semántico general por persona — DEC-014/DEC-015) y compararlo con el clustering estructural de `06_clustering.ipynb` (13 subperfiles sobre `X_modelado.csv`, features tabulares).

**Por qué:** sugerencia del profesor — clusterizar directamente en el espacio de embeddings valida si los perfiles estructurales son un patrón real (que también emerge desde el texto narrativo de la trayectoria/formación/docencia/investigación de cada persona) o un artefacto de la ingeniería de features tabular. Es una **capa de comparación/validación adicional**, no un reemplazo: el clustering estructural (`06_clustering`) sigue siendo la base de los perfiles del dashboard, interpretable variable por variable — este nuevo clustering vive en un espacio latente (1024 dimensiones de `BAAI/bge-m3`, DEC-028) sin significado individual por dimensión.

**Metodología (misma que `06_clustering.ipynb`, sección 2-3):** se evalúa un rango amplio de K (2-25) con silhouette/Calinski-Harabasz/Davies-Bouldin combinados con estabilidad (ARI entre semillas), descartando K<8 (particiones triviales, ver sección 4) para dar espacio a **más granularidad útil para negocio** en vez de forzar el corte más estable — resultado: **K=8** (DEC-028: embeddings regenerados con `BAAI/bge-m3`, espacio de 1024 dims; K=9 fue el resultado con el modelo anterior, `intfloat/multilingual-e5-base`). El modelo final se reentrena con `KMeans` completo (`n_init=10`); el escaneo/estabilidad usan `MiniBatchKMeans` por rendimiento (ver nota en sección 2).

**Interpretación de los clusters resultantes:** como el espacio de embeddings no tiene variables individuales interpretables, cada cluster se caracteriza post-hoc con las variables **estructuradas ya existentes** de `personas_dashboard.csv` — el título de cada perfil se construye automáticamente (tipo de empleado + cargo más frecuente + rasgo de trayectoria/producción académica más distintivo), sin nombrado manual todavía.

**Nota de reproducibilidad:** en este entorno, ejecutar este notebook completo vía `nbclient`/Jupyter tomó un tiempo desproporcionado (>30 min) frente al mismo código corrido como script plano (~3 min) — un problema de overhead del kernel Jupyter en Windows, no del cómputo en sí (confirmado con timing por celda). Los resultados y outputs de este notebook fueron generados corriendo el mismo código como script y sincronizados aquí; si se re-ejecuta el notebook completo desde Jupyter, considerar correrlo por celdas en vez de "Run All" si resulta lento.

**Salidas** (`data/clustering/*_semantico.csv` y `data/dashboard/*_semantico.csv`), mismo esquema de columnas que sus equivalentes estructurales para poder reutilizar el código del dashboard:
- `evaluacion_k_semantico.csv`, `clusters_personas_semantico.csv`, `pca_personas_semantico.csv`, `comparacion_ari_estructural_vs_semantico.csv` (en `data/clustering/`)
- `cluster_perfiles_resumen_semantico.csv`, `cluster_top_features_semantico.csv` (en `data/dashboard/`)

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

ROOT = Path.cwd().parents[1] if (Path.cwd().name == "07b_clustering_semantico") else Path.cwd()
DATA_EMBEDDINGS = ROOT / "data" / "embeddings"
DATA_CLUSTERING = ROOT / "data" / "clustering"
DATA_DASHBOARD = ROOT / "data" / "dashboard"
DATA_CLUSTERING.mkdir(parents=True, exist_ok=True)
DATA_DASHBOARD.mkdir(parents=True, exist_ok=True)

print("Directorio de trabajo:", ROOT)
print("Salidas de clustering (K, PCA, comparacion) en:", DATA_CLUSTERING)
print("Salidas de dashboard (resumen, top features) en:", DATA_DASHBOARD)

## 1. Carga de datos

Se usa el embedding **general** (`embeddings_personas.csv`, documento semántico completo: trayectoria + formación + docencia + investigación + vinculación + capacitación + idiomas + reconocimientos — DEC-014), no el embedding de trayectoria (`embeddings_trayectoria.csv`, más angosto y con menor cobertura de población). El embedding general es el más rico y el más comparable en alcance con el clustering estructural.

In [ ]:
emb_df = pd.read_csv(DATA_EMBEDDINGS / "embeddings_personas.csv")
ids = emb_df["IDPERSONA"].to_numpy()
X = emb_df.drop(columns=["IDPERSONA"]).to_numpy(dtype=np.float32)

# Normalizacion L2 (igual que lib.load_embeddings): el modelo e5 se entrena/usa con
# similitud coseno, asi que K-Means sobre vectores normalizados es equivalente a
# agrupar por similitud coseno (K-Means euclidiano sobre vectores L2-normalizados
# ordena igual que 1 - similitud_coseno).
normas = np.linalg.norm(X, axis=1, keepdims=True)
normas[normas == 0] = 1.0
X = X / normas

personas_dashboard = pd.read_csv(DATA_DASHBOARD / "personas_dashboard.csv", low_memory=False)

print(f"Personas con embedding: {len(ids)}")
print(f"Dimensiones del embedding: {X.shape[1]}")
print(f"Personas en personas_dashboard.csv: {len(personas_dashboard)}")

## 2. Selección del número de clusters (K)

Mismo procedimiento que `06_clustering.ipynb` sección 2: se evalúan varios K y se registran silhouette, Calinski-Harabasz, Davies-Bouldin y tamaños de cluster. No se fuerza K=13 (el K del clustering estructural) — se elige el mejor K propio de este espacio.

**Nota de rendimiento:** el escaneo (este paso) y la estabilidad (sección 3) usan `MiniBatchKMeans` en vez de `KMeans` completo — con 3194 personas × 1024 dimensiones y un rango amplio de K (2-25) más varias semillas por K, `KMeans(n_init=10)` es demasiado lento para explorar. `MiniBatchKMeans` da resultados equivalentes para elegir K (mismo objetivo, converge a soluciones muy similares) a una fracción del costo. El modelo **final** (sección 4 en adelante, una vez decidido K) sí se reentrena con `KMeans` completo (`n_init=10`) para quedarse con la mejor asignación posible.

In [ ]:
K_RANGE = range(2, 26)

filas_k = []
labels_por_k = {}

for k in K_RANGE:
    km = MiniBatchKMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE, batch_size=512)
    labels = km.fit_predict(X)
    labels_por_k[k] = labels
    sizes = pd.Series(labels).value_counts()
    filas_k.append({
        "K": k,
        "SILHOUETTE": silhouette_score(X, labels, metric="cosine"),
        "CALINSKI_HARABASZ": calinski_harabasz_score(X, labels),
        "DAVIES_BOULDIN": davies_bouldin_score(X, labels),
        "MIN_CLUSTER_SIZE": int(sizes.min()),
        "MAX_CLUSTER_SIZE": int(sizes.max()),
    })

evaluacion_k = pd.DataFrame(filas_k)
evaluacion_k.to_csv(DATA_CLUSTERING / "evaluacion_k_semantico.csv", index=False)
evaluacion_k

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(evaluacion_k["K"], evaluacion_k["SILHOUETTE"], marker="o")
axes[0, 0].set_title("Silhouette coseno (mayor es mejor)")
axes[0, 0].set_xlabel("K")

axes[0, 1].plot(evaluacion_k["K"], evaluacion_k["CALINSKI_HARABASZ"], marker="o", color="#55A868")
axes[0, 1].set_title("Calinski-Harabasz (mayor es mejor)")
axes[0, 1].set_xlabel("K")

axes[1, 0].plot(evaluacion_k["K"], evaluacion_k["DAVIES_BOULDIN"], marker="o", color="#C44E52")
axes[1, 0].set_title("Davies-Bouldin (menor es mejor)")
axes[1, 0].set_xlabel("K")

axes[1, 1].plot(evaluacion_k["K"], evaluacion_k["MIN_CLUSTER_SIZE"], marker="o", label="mínimo")
axes[1, 1].plot(evaluacion_k["K"], evaluacion_k["MAX_CLUSTER_SIZE"], marker="o", label="máximo")
axes[1, 1].set_title("Tamaño de clusters")
axes[1, 1].set_xlabel("K")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 3. Estabilidad de K-Means (ARI entre semillas)

Mismo criterio que `06_clustering.ipynb` sección 3: para cada K candidato se corre K-Means con varias semillas y se mide el Adjusted Rand Index (ARI) entre particiones — un ARI cercano a 1 indica que el K es robusto a la inicialización aleatoria, no un artefacto de una semilla en particular.

In [ ]:
SEMILLAS = [0, 1, 2, 42, 123]
K_CANDIDATOS_ESTABILIDAD = [k for k in K_RANGE if evaluacion_k.set_index("K").loc[k, "MIN_CLUSTER_SIZE"] >= 20]

filas_estabilidad = []
for k in K_CANDIDATOS_ESTABILIDAD:
    labels_semillas = []
    for s in SEMILLAS:
        km = MiniBatchKMeans(n_clusters=k, n_init=10, random_state=s, batch_size=512)
        labels_semillas.append(km.fit_predict(X))
    aris = [
        adjusted_rand_score(labels_semillas[i], labels_semillas[j])
        for i in range(len(labels_semillas)) for j in range(i + 1, len(labels_semillas))
    ]
    filas_estabilidad.append({"K": k, "ARI_PROMEDIO": np.mean(aris), "ARI_MIN": np.min(aris)})

estabilidad_k = pd.DataFrame(filas_estabilidad)
estabilidad_k

## 4. Selección del K final

Se elige el K con mejor combinación de silhouette + estabilidad (ARI promedio alto) entre los candidatos con tamaño de cluster mínimo razonable (≥20 personas) — mismo criterio de descarte que el clustering estructural (DEC-001: K con `min_cluster_size` muy bajo produce clusters de ruido, no perfiles útiles).

**Solo compiten K≥8**: igual que en `06_clustering.ipynb` (donde K=2 se descartó explícitamente por ser "una partición demasiado agregada... dominada por el volumen general de actividad/antigüedad, no por el objetivo de perfiles multidimensionales útiles"), los K bajos (2-7) en este espacio tienen el ARI más alto (≈0.7–0.998) precisamente porque son particiones triviales que casi cualquier semilla encuentra igual — no porque sean la partición más informativa (ver sección 7: con K=2, perfiles estructurales completamente distintos, p. ej. "Autoridad académica" 36–0, caen casi enteros en un solo cluster semántico). Se busca deliberadamente **más granularidad útil para el negocio** (más perfiles distinguibles, no solo el corte más estable), así que la selección automática solo mira K≥8 — el mejor K sigue siendo el que gane silhouette+estabilidad, pero dentro de un rango que ya garantiza varios perfiles.

In [ ]:
resumen_seleccion = evaluacion_k.merge(estabilidad_k, on="K", how="inner")
resumen_seleccion = resumen_seleccion[resumen_seleccion["K"] >= 8]  # ver nota de arriba: granularidad util para negocio
resumen_seleccion["SCORE"] = resumen_seleccion["SILHOUETTE"].rank(pct=True) + resumen_seleccion["ARI_PROMEDIO"].rank(pct=True)
resumen_seleccion = resumen_seleccion.sort_values("SCORE", ascending=False)
display(resumen_seleccion)

K_FINAL = int(resumen_seleccion.iloc[0]["K"])
print(f"\nK elegido: {K_FINAL}")

**Nota:** si `K_FINAL` no es el K que se prefiere tras revisar la tabla/gráficos de arriba, se puede sobreescribir manualmente en la celda siguiente antes de continuar.

In [ ]:
# K_FINAL = 8  # descomentar y ajustar si se prefiere otro K tras revisar la seccion 4

modelo_final = KMeans(n_clusters=K_FINAL, n_init=10, random_state=RANDOM_STATE)
cluster_labels = modelo_final.fit_predict(X)

clusters_personas_semantico = pd.DataFrame({"IDPERSONA": ids, "CLUSTER_SEMANTICO": cluster_labels})
clusters_personas_semantico.to_csv(DATA_CLUSTERING / "clusters_personas_semantico.csv", index=False)

cluster_sizes = clusters_personas_semantico["CLUSTER_SEMANTICO"].value_counts().sort_index()
print(f"K final: {K_FINAL}")
print(cluster_sizes)

## 5. PCA 2D (solo visualización)

Igual principio que `06_clustering.ipynb` sección 10: el clustering se hizo sobre las 1024 dimensiones completas del embedding; esta proyección PCA a 2D es únicamente una ayuda visual para el mapa del dashboard, no resume fielmente el espacio completo.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
var_explicada = pca.explained_variance_ratio_

pca_personas_semantico = pd.DataFrame({"IDPERSONA": ids, "PC1": X_pca[:, 0], "PC2": X_pca[:, 1]})
pca_personas_semantico.to_csv(DATA_CLUSTERING / "pca_personas_semantico.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 6))
paleta = sns.color_palette("Set2", K_FINAL)
for c in range(K_FINAL):
    mask = cluster_labels == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=12, alpha=0.6, color=paleta[c], label=f"Cluster {c}")
ax.set_xlabel(f"PC1 ({var_explicada[0]:.1%} var. explicada)")
ax.set_ylabel(f"PC2 ({var_explicada[1]:.1%} var. explicada)")
ax.set_title("Clusters semánticos (embeddings) proyectados en 2D con PCA")
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plt.show()

print(f"Varianza explicada por las 2 primeras componentes: {var_explicada.sum():.1%}")

## 6. Caracterización de cada cluster con variables estructuradas

El espacio de embeddings no tiene variables individuales interpretables (no hay un "eje" que signifique "nivel académico" o "tipo de empleado"). Para poder describir qué agrupó cada cluster, se cruza la asignación de cluster con las variables **ya existentes** de `personas_dashboard.csv` — mismo principio que interpretar un cluster de texto por quién cayó dentro, no por las coordenadas del vector.

In [ ]:
import re
from collections import Counter

df = clusters_personas_semantico.merge(personas_dashboard, on="IDPERSONA", how="left")
N_TOTAL = len(df)

COLUMNAS_NUMERICAS_PERFIL = [
    "ANTIGUEDAD_EFECTIVA_ANIOS", "NUM_PUBLICACIONES", "NUM_PROYECTOS_INVESTIGACION",
    "NUM_PERIODOS_DOCENCIA", "NUM_CAPACITACIONES", "N_UNIDADES_DISTINTAS", "N_CARGOS_DISTINTOS",
    "ANIOS_EXPERIENCIA_DOCENTE", "ANIOS_EXPERIENCIA_ADMINISTRATIVO",
]

# Etiquetas legibles para nivel academico (el dato crudo viene en snake_case: cuarto_nivel,
# tercer_nivel, etc.) - solo para construir el titulo del perfil, no se persiste como una
# nueva categoria institucional.
ETIQUETAS_NIVEL = {
    "cuarto_nivel": "posgrado", "tercer_nivel": "tercer nivel",
    "bachillerato": "bachillerato", "sin_registro": "sin nivel registrado",
}

# ---------------------------------------------------------------------------
# Nombrado por EVIDENCIA TEXTUAL (no solo metadata tabular): un titulo construido solo con
# variables estructuradas (cargo mas frecuente/antiguedad/nivel) puede ser enganoso cuando el
# patron real que agrupo al cluster vive en el TEXTO del documento semantico (unidad
# institucional predominante, tipo de trayectoria narrada, o ausencia total de trayectoria en
# los casos que el embedding se domino por FORMACION) - ver hallazgo manual: un cluster de
# perfiles con doctorado/maestria terminaba titulado "de ingreso reciente" porque sus
# documentos top no tienen seccion Trayectoria, y la regla tabular caia al fallback de
# antiguedad=0. Este bloque identifica, para cada cluster, los documentos MAS CERCANOS AL
# CENTROIDE del propio embedding (no una muestra aleatoria) y extrae de ahi el cargo/unidad
# del PRIMER evento de Trayectoria de cada uno (el mas reciente, ver PATRON_PRIMER_EVENTO
# mas abajo - contar todos los eventos de la trayectoria completa mezcla roles pasados con el
# actual y diluye la senal), excluyendo cargos genericos tipo "SERVICIOS PROFESIONALES -
# EJECUCION DE ACTIVIDADES" que aparecen en casi todos los clusters y no aportan senal
# distintiva.
#
# UMBRAL_DOMINANCIA: un cluster puede no tener una sola unidad/cargo claramente mayoritaria
# entre sus documentos representativos (ej. un cluster de docentes ocasionales repartido casi
# por igual entre 3-4 facultades: FCNM 38%, FIEC 24%, otras). Poner esa unica unidad ganadora
# en el TITULO en ese caso es enganoso - sugiere una concentracion que no existe. Si el top-1
# no alcanza el 40% de los documentos con match, el titulo omite la unidad/cargo especifico y
# usa una frase generica ("en varias unidades académicas"); el desglose completo (top 4 con
# porcentaje real) se guarda aparte en DESGLOSE_UNIDAD_TEXTUAL/DESGLOSE_CARGO_TEXTUAL para
# quien necesite el detalle exacto, sin forzarlo dentro del nombre corto.
UMBRAL_DOMINANCIA = 0.40
N_EJEMPLOS_CENTROIDE = 30  # cuantos documentos mas cercanos al centroide se inspeccionan para extraer patrones
N_EJEMPLOS_MOSTRADOS = 8   # cuantos de esos se guardan como evidencia citable en el CSV (EJEMPLOS_TEXTO)

CARGOS_GENERICOS = {
    "SERVICIOS PROFESIONALES - EJECUCIÓN DE ACTIVIDADES",
    "SERVICIOS PROFESIONALES - EJECUCION DE ACTIVIDADES",
}
# Captura el PRIMER evento de la seccion Trayectoria (el mas reciente/vigente, por
# construccion del texto en _seccion_trayectoria) - no todos los eventos del documento:
# contar cargo/unidad sobre TODOS los eventos mezcla trayectorias pasadas con la actual y
# diluye la senal (ej. un profesor ocasional que en algun momento tambien paso por
# Admisiones terminaba clasificado por esa unidad de paso, no por su rol dominante real).
# Tolera un sufijo opcional de jornada "(TC)/(TP)/(MT)" entre el cargo y "en" (ej. "PROFESOR
# NO TITULAR OCASIONAL (TC) en FACULTAD..."), y capta tanto "UNIDAD (SIGLAS)" como
# "DESCONOCIDA" (trayectorias historicas sin unidad registrada en el dato crudo -
# "DESCONOCIDA" en si es una senal real y distintiva, no debe descartarse silenciosamente).
PATRON_PRIMER_EVENTO = re.compile(
    r"^Trayectoria:\s*(?:Fue|Es)\s+([A-ZÁÉÍÓÚÑ0-9 \-]+?)(?:\s*\([A-ZÁÉÍÓÚÑ]{1,4}\))?\s+en\s+"
    r"(DESCONOCIDA|[A-ZÁÉÍÓÚÑ0-9 \-,]+?\s*\([A-ZÁÉÍÓÚÑ]+\))"
)
PATRON_FORMACION_ALTA = re.compile(r"Doctor|Ph\.?D|Magi?ster|Master", re.IGNORECASE)
# Primer titulo mencionado en la seccion Formacion academica (formato "Titulo - Institucion -
# Anio", ver _seccion_formacion en _embeddings_comun.py) - usado solo para MEDIR que tan
# concentrada esta esta seccion vs. Trayectoria, no para construir el nombre (ver comentario
# mas abajo en _evidencia_textual_cluster).
PATRON_FORMACION_TITULO = re.compile(r"Formación académica:\s*([^-\n]+?)\s*-\s*[^-\n]+?\s*-\s*\d{4}")

documentos_semanticos = pd.read_csv(DATA_EMBEDDINGS / "documento_semantico_persona.csv", usecols=["IDPERSONA", "DOCUMENTO_TEXTO"])

# Mapa SIGLAS -> nombre completo de unidad (misma fuente institucional que
# construir_mapa_siglas_unidad en _preprocesamiento_comun.py, DEC-022, pero en direccion
# inversa: aqui se necesita mostrar el nombre completo a partir de la sigla extraida del
# texto, no al reves).
_registro_autoridades = pd.read_csv(ROOT / "data" / "processed" / "registro_autoridades.csv", usecols=["SIGLAS", "NOMBRE"])
MAPA_SIGLAS_A_NOMBRE = dict(
    _registro_autoridades.dropna().drop_duplicates(subset=["SIGLAS"])[["SIGLAS", "NOMBRE"]].to_numpy()
)
MAPA_SIGLAS_A_NOMBRE["DESCONOCIDA"] = "unidad no registrada"


def _evidencia_textual_cluster(idpersonas_cluster: set[int], centroide: np.ndarray) -> dict:
    """Devuelve cargo/unidad predominantes (extraidos del texto real, no de metadata) y una
    lista de fragmentos de ejemplo, a partir de los documentos mas cercanos al centroide del
    cluster en el espacio de embeddings."""
    idx_cluster = np.array([i for i, idp in enumerate(ids) if idp in idpersonas_cluster])
    sims = X[idx_cluster] @ centroide
    orden = np.argsort(-sims)
    ids_top = [ids[idx_cluster[i]] for i in orden[:N_EJEMPLOS_CENTROIDE]]

    textos_top = (
        documentos_semanticos.set_index("IDPERSONA")
        .reindex(ids_top)["DOCUMENTO_TEXTO"]
        .dropna()
        .tolist()
    )

    cargos = Counter()
    unidades = Counter()
    titulos_formacion = Counter()
    n_con_formacion_alta = 0
    n_con_trayectoria = 0
    for txt in textos_top:
        txt = str(txt)
        if txt.startswith("Trayectoria:"):
            n_con_trayectoria += 1
        m = PATRON_PRIMER_EVENTO.match(txt)
        if m:
            cargo = m.group(1).strip()
            if cargo.upper() not in CARGOS_GENERICOS:
                cargos[cargo] += 1
            siglas_match = re.search(r"\(([A-ZÁÉÍÓÚÑ]+)\)", m.group(2))
            unidades[siglas_match.group(1) if siglas_match else "DESCONOCIDA"] += 1
        if PATRON_FORMACION_ALTA.search(txt):
            n_con_formacion_alta += 1
        # Comparacion honesta con evidencia textual (ver pregunta del usuario: "por que se
        # basa solo en trayectoria y no en formacion/otras secciones?"): se mide tambien la
        # concentracion real del primer titulo de FORMACION en los mismos 30 documentos, para
        # poder mostrar en el dashboard que Trayectoria gana porque tiene mas senal real (no
        # por sesgo del codigo) - Formacion tipicamente da 7-13% de concentracion (cada
        # persona tiene un titulo distinto) frente al 40-100% que suele dar Trayectoria.
        m_form = PATRON_FORMACION_TITULO.search(txt)
        if m_form:
            titulos_formacion[m_form.group(1).strip()] += 1

    ejemplos = [
        (str(t)[:220] + "...") if len(str(t)) > 220 else str(t)
        for t in textos_top[:N_EJEMPLOS_MOSTRADOS]
    ]

    total_cargo = sum(cargos.values())
    total_unidad = sum(unidades.values())
    cargo_top1, cargo_top1_n = (cargos.most_common(1) or [(None, 0)])[0]
    unidad_top1, unidad_top1_n = (unidades.most_common(1) or [(None, 0)])[0]
    pct_cargo_top1 = cargo_top1_n / total_cargo if total_cargo else 0.0
    pct_unidad_top1 = unidad_top1_n / total_unidad if total_unidad else 0.0

    def _desglose(counter: Counter, total: int, mapa_nombre: dict | None = None) -> str:
        if not total:
            return ""
        partes = []
        for clave, n in counter.most_common(4):
            etiqueta = mapa_nombre.get(clave, clave) if mapa_nombre else clave
            partes.append(f"{etiqueta} ({n / total:.0%})")
        return "; ".join(partes)

    return {
        "cargo_textual": cargo_top1,
        "unidad_textual": unidad_top1,
        "cargo_textual_dominante": cargo_top1 if pct_cargo_top1 >= UMBRAL_DOMINANCIA else None,
        "unidad_textual_dominante": unidad_top1 if pct_unidad_top1 >= UMBRAL_DOMINANCIA else None,
        "pct_cargo_top1": round(pct_cargo_top1, 3) if total_cargo else None,
        "pct_unidad_top1": round(pct_unidad_top1, 3) if total_unidad else None,
        "desglose_cargo": _desglose(cargos, total_cargo),
        "desglose_unidad": _desglose(unidades, total_unidad, MAPA_SIGLAS_A_NOMBRE),
        "pct_con_trayectoria": n_con_trayectoria / len(textos_top) if textos_top else None,
        "pct_con_formacion_alta": n_con_formacion_alta / len(textos_top) if textos_top else None,
        # Concentracion del titulo de FORMACION mas repetido vs. la del cargo/unidad de
        # TRAYECTORIA, en los mismos 30 documentos - permite mostrar en el dashboard por que
        # el nombre se basa en trayectoria (evidencia real, no una eleccion arbitraria del
        # codigo): normalmente Formacion da 7-13% (cada persona tiene un titulo distinto) muy
        # por debajo del 40-100% que da Trayectoria cuando hay una unidad/cargo dominante.
        "pct_formacion_titulo_top1": round(
            titulos_formacion.most_common(1)[0][1] / sum(titulos_formacion.values()), 3
        ) if titulos_formacion else None,
        "n_formacion_con_match": sum(titulos_formacion.values()),
        "ejemplos_texto": " || ".join(ejemplos),
    }


def _rasgo_trayectoria(mediana_cluster: dict, mediana_global: dict) -> str | None:
    """Identifica el rasgo mas distintivo del cluster (trayectoria/antiguedad primero,
    produccion academica despues) cuando el cargo mas frecuente por si solo NO diferencia
    (varios clusters pueden compartir el mismo "cargo mas frecuente" pero representar
    patrones de carrera muy distintos: recien ingresados vs. trayectoria larga con muchos
    cargos, o con vs. sin produccion academica) - evita titulos identicos entre clusters.

    Orden de prioridad: primero el rasgo de ANTIGUEDAD/MULTI-ROL si es muy extremo (se
    revisa antes que investigacion/publicaciones porque un cluster puede tener AMBOS rasgos
    a la vez, y la antiguedad/multi-rol es la señal mas fuerte y menos ambigua para
    diferenciar patrones de carrera); solo si no hay nada extremo en antiguedad, se cae a
    produccion academica. Devuelve None si nada se aleja lo suficiente de la mediana global."""
    antig = mediana_cluster.get("ANTIGUEDAD_EFECTIVA_ANIOS")
    antig_global = mediana_global.get("ANTIGUEDAD_EFECTIVA_ANIOS")
    n_cargos = mediana_cluster.get("N_CARGOS_DISTINTOS")
    n_cargos_global = mediana_global.get("N_CARGOS_DISTINTOS")
    pub = mediana_cluster.get("NUM_PUBLICACIONES")
    proy = mediana_cluster.get("NUM_PROYECTOS_INVESTIGACION")

    if (
        pd.notna(antig) and pd.notna(antig_global) and pd.notna(n_cargos) and pd.notna(n_cargos_global)
        and antig >= antig_global * 3 and n_cargos >= n_cargos_global * 3
    ):
        return "de trayectoria larga y multi-rol"
    if pd.notna(pub) and pub >= 1:
        return "con producción académica (publicaciones/proyectos)"
    if pd.notna(proy) and proy >= 1 and (pd.isna(pub) or pub == 0):
        return "con participación en investigación"
    if pd.notna(antig) and pd.notna(antig_global):
        if antig <= antig_global * 0.3:
            return "de ingreso reciente"
        if antig >= antig_global * 1.5:
            return "de trayectoria consolidada"
    return None


def _titulo_perfil(
    tipo_top: pd.Series, cargo_desc: str, nivel_top: pd.Series, pct_vigente: float,
    rasgo: str | None, evidencia: dict,
) -> str:
    """Construye el titulo del cluster priorizando la EVIDENCIA TEXTUAL (cargo/unidad
    extraidos de los documentos mas cercanos al centroide) sobre la metadata tabular, que se
    usa solo como fallback cuando el texto no aporta una senal clara de trayectoria (caso del
    cluster dominado por documentos de solo-FORMACION, ver `_evidencia_textual_cluster`)."""
    partes = []
    if len(tipo_top) and tipo_top.iloc[0] >= 0.55:
        partes.append(tipo_top.index[0].title())
    elif len(tipo_top):
        partes.append("Perfil mixto administrativo/docente")

    # Si menos de un tercio de los documentos representativos tienen seccion Trayectoria,
    # el cluster no se agrupo por cargo/unidad sino por otro contenido (tipicamente
    # Formacion) - un titulo de "cargo tipo X" seria enganoso, se usa un titulo distinto. En
    # este caso tampoco se usa `rasgo` (se calcula sobre ANTIGUEDAD_EFECTIVA_ANIOS, que sin
    # trayectoria da 0 y produciria el mismo error que motivo este cambio: un cluster con
    # perfiles de posgrado/doctorado terminaba titulado "de ingreso reciente").
    pct_con_trayectoria = evidencia.get("pct_con_trayectoria")
    sin_trayectoria_dominante = pct_con_trayectoria is not None and pct_con_trayectoria < (1 / 3)
    if sin_trayectoria_dominante:
        if evidencia.get("pct_con_formacion_alta", 0) >= 0.5:
            partes.append("con formación de posgrado (maestría/doctorado), sin trayectoria estructurada dominante")
        else:
            partes.append("sin trayectoria estructurada dominante en el texto")
    else:
        cargo_dom = evidencia.get("cargo_textual_dominante")
        unidad_dom = evidencia.get("unidad_textual_dominante")
        if cargo_dom and unidad_dom:
            unidad_legible = MAPA_SIGLAS_A_NOMBRE.get(unidad_dom, unidad_dom).title()
            partes.append(f"tipo {cargo_dom.title()} en {unidad_legible}")
        elif cargo_dom:
            # Cargo con dominancia clara pero repartido entre varias unidades (ninguna
            # supera UMBRAL_DOMINANCIA) - el detalle real vive en DESGLOSE_UNIDAD_TEXTUAL.
            partes.append(f"tipo {cargo_dom.title()} en varias unidades")
        elif unidad_dom:
            unidad_legible = MAPA_SIGLAS_A_NOMBRE.get(unidad_dom, unidad_dom).title()
            partes.append(f"predominantemente en {unidad_legible}")
        elif evidencia.get("cargo_textual") or evidencia.get("unidad_textual"):
            # Ni cargo ni unidad alcanzan el umbral de dominancia por separado: el cluster
            # esta genuinamente repartido, no forzar un titulo especifico enganoso.
            partes.append("con trayectoria repartida entre varios cargos/unidades")
        elif cargo_desc and cargo_desc != "sin dato":
            # Fallback a metadata tabular solo si la extraccion textual no encontro nada.
            partes.append(f"tipo {cargo_desc.title()}")

    if sin_trayectoria_dominante:
        if len(nivel_top):
            nivel_legible = ETIQUETAS_NIVEL.get(nivel_top.index[0], nivel_top.index[0])
            partes.append(f"nivel académico predominante: {nivel_legible}")
    elif rasgo:
        partes.append(rasgo)
    elif len(nivel_top):
        nivel_legible = ETIQUETAS_NIVEL.get(nivel_top.index[0], nivel_top.index[0])
        partes.append(f"con {nivel_legible}")

    if pd.notna(pct_vigente) and pct_vigente >= 0.6:
        partes.append("(mayoría vigente)")
    elif pd.notna(pct_vigente) and pct_vigente <= 0.15:
        partes.append("(mayoría no vigente)")

    return " ".join(partes) if partes else "Perfil sin caracterización clara"


medianas_global = {feat: df[feat].median() for feat in COLUMNAS_NUMERICAS_PERFIL if feat in df.columns}

filas_resumen = []
filas_top_features = []
for c in range(K_FINAL):
    sub = df[df["CLUSTER_SEMANTICO"] == c]
    n = len(sub)

    tipo_top = sub["TIPOEMPLEADO_ACTUAL_DESC"].value_counts(normalize=True)
    cargo_top = sub["CARGO_ACTUAL"].value_counts()
    nivel_top = sub["NIVEL_ACADEMICO_MAXIMO"].value_counts(normalize=True)
    pct_vigente = sub["VIGENTE_ACTUALMENTE"].astype("boolean").mean(skipna=True)

    tipo_desc = f"{tipo_top.index[0]} ({tipo_top.iloc[0]:.0%})" if len(tipo_top) else "sin dato"
    nivel_desc = f"{nivel_top.index[0]} ({nivel_top.iloc[0]:.0%})" if len(nivel_top) else "sin dato"
    cargo_desc = cargo_top.index[0] if len(cargo_top) else "sin dato"
    vigente_desc = f"{pct_vigente:.0%} vigente" if pd.notna(pct_vigente) else "vigencia sin dato"

    medianas_cluster = {feat: sub[feat].median() for feat in COLUMNAS_NUMERICAS_PERFIL if feat in sub.columns}
    rasgo = _rasgo_trayectoria(medianas_cluster, medianas_global)

    centroide_c = X[np.isin(ids, sub["IDPERSONA"].to_numpy())].mean(axis=0)
    centroide_c = centroide_c / np.linalg.norm(centroide_c)
    evidencia = _evidencia_textual_cluster(set(sub["IDPERSONA"]), centroide_c)

    titulo = _titulo_perfil(tipo_top, cargo_desc, nivel_top, pct_vigente, rasgo, evidencia)
    descripcion = (
        f"{n} personas ({n / N_TOTAL:.1%} de la población con embedding). "
        f"Predominantemente {tipo_desc}, nivel académico más común {nivel_desc}, "
        f"cargo más frecuente: {cargo_desc}. {vigente_desc}."
    )

    filas_resumen.append({
        "CLUSTER_SEMANTICO": c,
        "PERFIL_NOMBRE_SEMANTICO": titulo,
        "N_PERSONAS": n,
        "PCT_POBLACION": round(100 * n / N_TOTAL, 2),
        "DESCRIPCION": descripcion,
        "TIPOEMPLEADO_PREDOMINANTE": tipo_top.index[0] if len(tipo_top) else None,
        "CARGO_MAS_FRECUENTE": cargo_desc,
        "NIVEL_ACADEMICO_PREDOMINANTE": nivel_top.index[0] if len(nivel_top) else None,
        "PCT_VIGENTE": round(100 * pct_vigente, 1) if pd.notna(pct_vigente) else None,
        "CARGO_TEXTUAL_PREDOMINANTE": evidencia.get("cargo_textual"),
        "UNIDAD_TEXTUAL_PREDOMINANTE": evidencia.get("unidad_textual"),
        "PCT_CARGO_TEXTUAL_TOP1": evidencia.get("pct_cargo_top1"),
        "PCT_UNIDAD_TEXTUAL_TOP1": evidencia.get("pct_unidad_top1"),
        "DESGLOSE_CARGO_TEXTUAL": evidencia.get("desglose_cargo"),
        "DESGLOSE_UNIDAD_TEXTUAL": evidencia.get("desglose_unidad"),
        "PCT_CON_TRAYECTORIA": round(100 * evidencia["pct_con_trayectoria"], 1) if evidencia.get("pct_con_trayectoria") is not None else None,
        "PCT_FORMACION_TITULO_TOP1": round(100 * evidencia["pct_formacion_titulo_top1"], 1) if evidencia.get("pct_formacion_titulo_top1") is not None else None,
        "EJEMPLOS_TEXTO": evidencia.get("ejemplos_texto"),
    })

    for feat in COLUMNAS_NUMERICAS_PERFIL:
        if feat not in df.columns:
            continue
        val_cluster = sub[feat].median()
        val_global = df[feat].median()
        if pd.isna(val_cluster) or pd.isna(val_global):
            continue
        filas_top_features.append({
            "CLUSTER_SEMANTICO": c, "FEATURE": feat,
            "VALUE_CLUSTER": round(float(val_cluster), 2), "VALUE_GLOBAL": round(float(val_global), 2),
            "DIFERENCIA_ABS": round(abs(float(val_cluster) - float(val_global)), 2),
        })

cluster_perfiles_resumen_semantico = pd.DataFrame(filas_resumen)

# Si dos o mas clusters igual quedaron con el mismo titulo (mismo cargo+rasgo), se
# desambiguan agregando un sufijo numerico - nunca deben mostrarse dos perfiles identicos en
# el dashboard (Resumen/Mapa), aunque el rasgo de trayectoria ya reduce mucho esa colision.
duplicados = cluster_perfiles_resumen_semantico["PERFIL_NOMBRE_SEMANTICO"].duplicated(keep=False)
if duplicados.any():
    contador = {}
    for idx in cluster_perfiles_resumen_semantico[duplicados].index:
        nombre = cluster_perfiles_resumen_semantico.loc[idx, "PERFIL_NOMBRE_SEMANTICO"]
        contador[nombre] = contador.get(nombre, 0) + 1
        cluster_perfiles_resumen_semantico.loc[idx, "PERFIL_NOMBRE_SEMANTICO"] = f"{nombre} ({contador[nombre]})"

cluster_perfiles_resumen_semantico.to_csv(DATA_DASHBOARD / "cluster_perfiles_resumen_semantico.csv", index=False)

# Top 6 features mas distintivas por cluster (mayor diferencia absoluta vs. mediana global)
cluster_top_features_semantico = (
    pd.DataFrame(filas_top_features)
    .sort_values(["CLUSTER_SEMANTICO", "DIFERENCIA_ABS"], ascending=[True, False])
    .groupby("CLUSTER_SEMANTICO")
    .head(6)
    .drop(columns="DIFERENCIA_ABS")
)
cluster_top_features_semantico.to_csv(DATA_DASHBOARD / "cluster_top_features_semantico.csv", index=False)

cluster_perfiles_resumen_semantico[
    ["CLUSTER_SEMANTICO", "PERFIL_NOMBRE_SEMANTICO", "N_PERSONAS", "PCT_POBLACION", "DESCRIPCION"]
]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
orden = cluster_perfiles_resumen_semantico.sort_values("CLUSTER_SEMANTICO")
bars = ax.bar(orden["CLUSTER_SEMANTICO"].astype(str), orden["N_PERSONAS"], color=sns.color_palette("Set2", len(orden)))
for bar, pct in zip(bars, orden["PCT_POBLACION"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5, f"{pct:.1f}%", ha="center")
ax.set_xlabel("Cluster semántico")
ax.set_ylabel("N° de personas")
ax.set_title(f"Tamaño de los {K_FINAL} clusters semánticos (embeddings)")
plt.tight_layout()
plt.show()

## 7. Comparación contra el clustering estructural (evidencia para el profesor)

Se cruza la asignación de cluster semántico (este notebook) contra el `CLUSTER` estructural (`06_clustering.ipynb`, 13 subperfiles) para la población en común, y se calcula ARI/NMI — la misma comparación que ya se hizo una vez entre embeddings y clusters estructurales (ver `context/DECISION_LOG.md`, resultado previo ARI≈0.116, señal de que ambos espacios capturan patrones distintos y complementarios, no redundantes).

In [ ]:
comparacion = df[df["CLUSTER"] != -1][["IDPERSONA", "CLUSTER", "CLUSTER_SEMANTICO", "PERFIL_NOMBRE"]].dropna(subset=["CLUSTER"])
comparacion["CLUSTER"] = comparacion["CLUSTER"].astype(int)

ari = adjusted_rand_score(comparacion["CLUSTER"], comparacion["CLUSTER_SEMANTICO"])
nmi = normalized_mutual_info_score(comparacion["CLUSTER"], comparacion["CLUSTER_SEMANTICO"])

print(f"Personas comparadas (con cluster estructural y semántico): {len(comparacion)}")
print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print(f"Normalized Mutual Information (NMI): {nmi:.4f}")
print("\n(ARI/NMI cercanos a 0 = particiones prácticamente independientes; cercanos a 1 = coinciden fuertemente.)")

tabla_contingencia = pd.crosstab(
    comparacion["PERFIL_NOMBRE"], comparacion["CLUSTER_SEMANTICO"],
    rownames=["Perfil estructural"], colnames=["Cluster semántico"],
)
tabla_contingencia.to_csv(DATA_CLUSTERING / "comparacion_ari_estructural_vs_semantico.csv")

fig, ax = plt.subplots(figsize=(1.2 * K_FINAL + 3, 0.5 * len(tabla_contingencia) + 2))
sns.heatmap(tabla_contingencia, annot=True, fmt="d", cmap="viridis", ax=ax, cbar_kws={"label": "N personas"})
ax.set_title(f"Perfil estructural (13) × Cluster semántico ({K_FINAL}) — ARI={ari:.3f}, NMI={nmi:.3f}")
plt.tight_layout()
plt.show()

## 8. Resumen final

Artefactos generados:

En `data/clustering/`:
- `evaluacion_k_semantico.csv` — métricas por K evaluado
- `clusters_personas_semantico.csv` — IDPERSONA → CLUSTER_SEMANTICO
- `pca_personas_semantico.csv` — proyección 2D para el mapa del dashboard
- `comparacion_ari_estructural_vs_semantico.csv` — tabla de contingencia contra los 13 perfiles estructurales

En `data/dashboard/` (mismo directorio que `cluster_perfiles_resumen.csv`/`cluster_top_features.csv` del clustering estructural):
- `cluster_perfiles_resumen_semantico.csv` — resumen por cluster (tamaño, descripción automática)
- `cluster_top_features_semantico.csv` — variables más distintivas por cluster (vs. mediana global)

**Siguiente paso:** exponer estos artefactos en el dashboard (Resumen general, Perfiles y Mapa) como una vista alternativa "Clustering en embeddings", claramente diferenciada del clustering estructural — ver `notebooks/08_dashboard/lib.py` y `dashboard_react/backend/main.py`.